# LangChain

In [26]:
# uv add -U langchain
# uv add -U langchain-openai
# uv add -U "langchain[google-genai]"
# uv add -U "langchain[anthropic]"

from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)

True

In [ ]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# 1. 기본 standalone model

In [5]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model='openai:gpt-5.4-mini',
    
    # [kwargs] 모델의 행동 설정:
    temperature=0.8,   # 창의성: 약간 창의적인 수준
    timeout=30,        # 인내심: 30초까지만 기다림
    max_tokens=100,   # 길이제한: 최대 1000 토큰까지만 출력
    max_retries=2      # 재시도: 실패 시 2번 더 시도함 (기본값은 보통 0~1)
)

result = model.invoke("대한민국의 수도는?")

print(result.content)

대한민국의 수도는 **서울**입니다.


In [6]:
# [실습 2] stream: 답변을 실시간으로 조각조각 받기
print("--- stream 시작 ---")
for chunk in model.stream("대한민국의 사계절에 대해 설명해줘"):
    # chunk.content에 글자 조각이 들어있음
    # end=""는 줄바꿈을 하지 않고 옆으로 붙여서 출력하라는 뜻
    print(chunk.content, end="", flush=True) 
print("\n--- stream 끝 ---")

--- stream 시작 ---
대한민국은 **사계절이 비교적 뚜렷한 온대 기후**를 가지고 있어요. 계절마다 날씨, 자연 풍경, 생활 방식이 크게 달라집니다.

## 1. 봄
- **시기:** 보통 3월~5월
- **특징:** 날씨가 따뜻해지고 꽃이 피기 시작해요.
- **기온:** 점차 올라가며,
--- stream 끝 ---


In [7]:
# [실습 3] batch: 질문 리스트 -> 답변 리스트
questions = [
    "사과를 영어로 뭐야?",
    "바나나를 영어로 뭐야?",
    "포도를 영어로 뭐야?"
]

print("--- batch 시작 (잠시 대기...) ---")
results = model.batch(questions)

# 결과 리스트 하나씩 출력
for res in results:
    print(f"답변: {res.content}")
print("--- batch 끝 ---")

--- batch 시작 (잠시 대기...) ---
답변: “사과”는 영어로 **apple**이에요.

원하시면 “사과하다”의 영어도 같이 알려드릴게요.
답변: 바나나는 영어로 **banana**예요.
답변: 포도는 영어로 **grape**예요.  
여러 개의 포도는 **grapes**라고 해요.
--- batch 끝 ---


In [9]:
from langchain.chat_models import init_chat_model

model = init_chat_model('google_genai:gemini-2.5-flash')

response = model.invoke('대한민국의 수도는?')
response.content

'대한민국의 수도는 **서울**입니다.'

# 2. Structured Outputs

## pydantic 형태

In [16]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class OrderInfo(BaseModel):
    name: str = Field(description='주문자 이름')
    address: str = Field(description='주문자 주소')
    order: list[str] = Field(description='주문한 음식')
    phone_num: str = Field(description='주문자 전화번호')

model = init_chat_model('gpt-5.4-mini')
structured_llm = model.with_structured_output(schema=OrderInfo)
result = structured_llm.invoke(
    '하와이안 피자말고 페페로니 피자랑 고구마 피자 각각 하나씩 서울시 용산구 용산빌딩 101호로 보내주세요. \
    제 이름은 한승헌이고 전화번호는 010-1234-1234입니다.'
)

#result["structured_response"]
result

OrderInfo(name='한승헌', address='서울시 용산구 용산빌딩 101호', order=['페페로니 피자', '고구마 피자'], phone_num='010-1234-1234')

In [18]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ProductReview(BaseModel):
    rating: int | None = Field(description="평점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감성")
    key_points: list[str] = Field(description="핵심 포인트, 소문자, 1~3단어")

agent = create_agent(
    model="gpt-5.4-mini",
    tools=[],
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "이 리뷰를 분석해줘: 배송은 빨랐지만 가격이 비쌌고 전체적으로 만족스러웠어요. 별점은 4점입니다."
        }
    ]
})

print(result["structured_response"])

rating=4 sentiment='positive' key_points=['배송', '가격', '만족']


In [17]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ProductReview(BaseModel):
    rating: int | None = Field(description="평점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감성")
    key_points: list[str] = Field(description="핵심 포인트, 소문자, 1~3단어")

model = init_chat_model('openai:gpt-5.4-mini')
structured_llm = model.with_structured_output(schema=ProductReview)
result = structured_llm.invoke('이 리뷰를 분석해줘: 배송은 빨랐지만 가격이 비쌌고, 전체적으로 만족스러웠어요. 별점은 4점입니다.')

result

ProductReview(rating=4, sentiment='positive', key_points=['빠른배송', '비싼가격', '전체만족'])

In [20]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class MeetingAction(BaseModel):
    task: str = Field(description="해야 할 일")
    assignee: str = Field(description="담당자")
    priority: Literal["low", "medium", "high"] = Field(description="우선순위")

agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="액션 아이템이 정리되었습니다."
    )
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "회의 내용: 영희가 이번 주 안에 발표 자료를 수정해야 하고 매우 급합니다."
        }
    ]
})

print(result["structured_response"])

task='이번 주 안에 발표 자료 수정 완료' assignee='영희' priority='high'


## TypedDict 형태

In [ ]:
import os
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model

class MovieInfo(TypedDict):
    title: str
    year: int
    genre: list[str]

model = init_chat_model("gpt-5.4-mini")

structured_model = model.with_structured_output(
    MovieInfo,
    method="json_schema"
)

result = structured_model.invoke("영화 인터스텔라 정보를 구조화해서 알려줘.")
print(result)

{'title': '인터스텔라 (Interstellar)', 'year': 2014, 'genre': ['SF', '모험', '드라마']}


# 3. 오류 처리 옵션 handle_errors

In [24]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy

class ProductRating(BaseModel):
    rating: int | None = Field(description="1~5 평점", ge=1, le=5)
    comment: str = Field(description="리뷰 코멘트")

response_format = ToolStrategy(
    schema=ProductRating,
    handle_errors="평점은 1~5 사이로 작성하고, comment도 꼭 포함하세요."
)

## json schema 정의

In [27]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# Gemini API Key 설정
# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"

# 1. JSON Schema 정의
product_rating_schema = {
    "title": "ProductRating",
    "description": "리뷰에서 평점과 코멘트를 추출하는 스키마",
    "type": "object",
    "properties": {
        "rating": {
            "type": "integer",
            "description": "1~5 사이의 평점",
            "minimum": 1,
            "maximum": 5
        },
        "comment": {
            "type": "string",
            "description": "리뷰에 대한 핵심 코멘트"
        }
    },
    "required": ["rating", "comment"]
}

# 2. Gemini 모델 생성
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

# 3. Structured Output 적용
structured_llm = llm.with_structured_output(
    schema=product_rating_schema,
    method="json_schema"
)

# 4. 실행
result = structured_llm.invoke(
    "다음 리뷰를 분석해서 평점과 코멘트를 추출해줘: 배송은 빨랐지만 포장이 조금 아쉬웠어요. 전체적으로는 만족합니다. 4점 정도예요."
)

print(result)

{'rating': 4, 'comment': '배송은 빨랐으나 포장이 아쉬웠지만 전체적으로 만족합니다.'}


### 실습

In [29]:
# exercise 4. 고객문의 분류기
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class CustomerInquiry(BaseModel):
    category: Literal['기술 지원', '결제/환불', '계정 관리', '기타'] = Field(description='문의 내용의 카테고리')
    urgency:  Literal["low", "medium", "high"] = Field(description="문의의 긴급도")
    needs_human_support: bool = Field(description='상담원의 직접적인 개입이 필요한지 여부')

agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[],
    response_format=ToolStrategy(
        schema=CustomerInquiry,
        tool_message_content="액션 아이템이 정리되었습니다."
    )
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "문의내용: 어제 결제했는데 아직 프리미엄 권한이 안 들어왔어요. 빨리 확인 좀 부탁드려요!"
        }
    ]
})

print(result["structured_response"])

category='결제/환불' urgency='high' needs_human_support=True


In [2]:
from typing import Literal, List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 1. 데이터 스키마 정의 (사용자 작성 기반 + 예외 처리 추가)
class NPCInfo(BaseModel):
    npc_name: str = Field(description='게임 NPC의 이름. 이름이 유추되지 않으면 "Unknown"으로 작성')
    emotion: Literal['joy', 'anger', 'sadness', 'fear', 'neutral'] = Field(description='해당 NPC의 감정')
    quest_related: bool = Field(description='NPC가 퀘스트 관련 언급을 하는지 여부')
    lore_tags: List[str] = Field(description='NPC가 알고 있는 지식이나 소속을 나타내는 태그(1~3개)')

# 2. 모델 설정 및 구조화된 출력 바인딩
# gpt-4o-mini가 정보 추출 가성비가 가장 좋습니다.
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
structured_llm = llm.with_structured_output(NPCInfo)

# 3. 프롬프트 템플릿 설계
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 게임 대사를 분석하여 NPC의 상태와 정보를 정확하게 추출하는 시스템입니다."),
    ("user", "대사: {dialogue}")
])

# 4. 체인 생성 (LCEL: 파이프라인 구조)
# 프롬프트의 결과가 모델의 입력으로 물 흐르듯 넘어갑니다.
chain = prompt | structured_llm

# ==========================================
# 5. 단일 대사 실행 (invoke)
# ==========================================
single_dialogue = """오, 외지에서 온 여행자로군. 이 **[북부_경계]**의 찬바람은 뼈저리게 시리니 조심하게나. 
겉보기엔 평화로운 광장이지만, 요즘 성안 분위기가 흉흉해. 
혹시 자네도 그 [반역자 수색] 때문에 온 건가? 내 눈은 속여도 내 귀는 못 속이지."""

print("--- [단일 추출 결과] ---")
result = chain.invoke({"dialogue": single_dialogue})
print(result)
# 출력: npc_name='Unknown' emotion='neutral' quest_related=True lore_tags=['북부_경계', '반역자 수색']

# ==========================================
# 6. 여러 대사 한 번에 처리 (batch) - LCEL의 강력한 기능
# ==========================================
dialogues_list = [
    {"dialogue": "내 이름은 바론. 여기서 대장장이 일을 하고 있지. 무기가 필요하면 언제든 말해."},
    {"dialogue": "히, 히익! 가, 가까이 오지 마세요! 그 폭발은 제가 한 게 아니라고요!"}
]

print("\n--- [다중(Batch) 추출 결과] ---")
batch_results = chain.batch(dialogues_list)

for idx, res in enumerate(batch_results):
    print(f"[{idx+1}] {res}")

--- [단일 추출 결과] ---
npc_name='Unknown' emotion='neutral' quest_related=True lore_tags=['북부_경계', '성안', '반역자 수색']

--- [다중(Batch) 추출 결과] ---
[1] npc_name='바론' emotion='neutral' quest_related=False lore_tags=['대장장이', '무기 제작', '상점 NPC']
[2] npc_name='Unknown' emotion='fear' quest_related=False lore_tags=[]


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. 모델 및 파서 설정
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.7)
output_parser = StrOutputParser() # LLM의 응답(AIMessage 객체)에서 텍스트만 깔끔하게 뽑아주는 역할

# ==========================================
# 체인 A: NPC의 '속마음'을 생성하는 프롬프트
# ==========================================
thought_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 게임 속 {npc_role}입니다. 현재 상황: {situation}"),
    ("user", "이 상황에서 당신이 속으로 무슨 생각을 할지 1~2문장의 독백으로 작성해.")
])

# 체인 A 조립
thought_chain = thought_prompt | llm | output_parser


# ==========================================
# 체인 B: 속마음을 바탕으로 실제 '대사'를 생성하는 프롬프트
# ==========================================
dialogue_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 게임 속 {npc_role}입니다."),
    ("user", """당신의 진짜 속마음은 '{inner_thought}' 이지만, 
    플레이어에게는 이 속마음을 적절히 숨기거나 포장해서 대화해야 합니다.
    게임 NPC처럼 몰입감 있는 대사 한 마디를 내뱉으세요.""")
])

# 체인 B 조립
dialogue_chain = dialogue_prompt | llm | output_parser


# ==========================================
# 🌟 최종 파이프라인 (LCEL의 꽃)
# ==========================================
# RunnablePassthrough.assign을 사용하면, 기존 입력값(npc_role, situation)을 유지한 채 
# thought_chain의 결과값을 'inner_thought'라는 새로운 키로 추가하여 다음 체인으로 넘깁니다.

full_chain = (
    RunnablePassthrough.assign(inner_thought=thought_chain) 
    | dialogue_chain
)

# 2. 실행
input_data = {
    "npc_role": "마을의 부유한 상인",
    "situation": "허름한 옷을 입은 초보 플레이어가 전설의 검을 팔겠다고 다가옴"
}

print("--- [대사 생성 중...] ---\n")

# 중간 과정을 확인하고 싶다면 아래와 같이 디버깅할 수도 있습니다.
thought = thought_chain.invoke(input_data)
print(f"💭 속마음: {thought}\n")

final_dialogue = full_chain.invoke(input_data)
print(f"🗣️ 실제 대사: {final_dialogue}")

--- [대사 생성 중...] ---

💭 속마음: “저 허름한 차림에 전설의 검이라니, 사기일 가능성이 높군. 하지만 진짜라면 한몫 크게 챙길 기회이니, 일단 표정 관리부터 해야겠어.”

🗣️ 실제 대사: 오호라, 그 전설의 검이라… 눈이 가는군요. 진품이라면 대단한 물건이겠지만, 세상엔 빛나는 헛소문도 많은 법이죠. 우선 한 번 자세히 보여주시겠습니까? 그러면 제가 그 가치를 정성껏 살펴보겠습니다.


In [ ]:
# memory에 대화내용을 기억하는 NPC : LangGraph를 사용해야함.
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

# 1. 상태(State) 정의: 파이프라인에서 굴러갈 데이터의 구조를 잡습니다.
class NPCState(TypedDict):
    npc_role: str
    situation: str
    inner_thought: str
    final_dialogue: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 2. 노드(Node) 함수 정의
def generate_thought(state: NPCState):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 게임 속 {npc_role}입니다. 현재 상황: {situation}"),
        ("user", "속으로 무슨 생각을 할지 1~2문장의 독백으로 작성해.")
    ])
    chain = prompt | llm
    response = chain.invoke(state)
    return {"inner_thought": response.content} # 상태 업데이트

def generate_dialogue(state: NPCState):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 게임 속 {npc_role}입니다."),
        ("user", "속마음 '{inner_thought}'을 적절히 숨기고 NPC처럼 대사해.")
    ])
    chain = prompt | llm
    response = chain.invoke(state)
    return {"final_dialogue": response.content} # 상태 업데이트

# 3. LangGraph 조립
workflow = StateGraph(NPCState)
workflow.add_node("think", generate_thought)
workflow.add_node("speak", generate_dialogue)

# 순서 연결: 시작 -> 생각 -> 말하기 -> 끝
workflow.set_entry_point("think")
workflow.add_edge("think", "speak")
workflow.add_edge("speak", END)

# 그래프 컴파일
npc_app = workflow.compile()

# 4. 실행
initial_state = {
    "npc_role": "마을의 부유한 상인",
    "situation": "허름한 옷을 입은 초보 플레이어가 전설의 검을 팔겠다고 다가옴"
}

# 스트리밍 형태로 중간 단계(속마음)를 확인하며 실행 가능
for output in npc_app.stream(initial_state):
    for key, value in output.items():
        if key == "think":
            print(f"💭 속마음: {value['inner_thought']}")
        elif key == "speak":
            print(f"\n🗣️ 실제 대사: {value['final_dialogue']}")